In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pcaDat = pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/KNC_091224_AADR_shrink.group.period.1811.csv',sep=',',
                    names=['Name','PC1','PC2','PC3','PC4','Population'],dtype={'Name':str},
                     skiprows=1, skipinitialspace=True)
print(pcaDat)

In [ ]:
popListDat = pd.read_csv('/mnt/archgen/users/xiaowen/Iraq_Syria/0325/PCA/0825/PCA_list.txt',names=['Population'])
CheckInd=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/0624/PCA/KNC_outlier.csv',names=['Individual','colorIndex','symbolIndex'],dtype={'Individual':object,'colorIndex':int,'symbolIndex':int})
refPop=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/KNCrefPop1_group_0425_simp.csv',names=['Individual','colorIndex','symbolIndex','CustomLabel'],dtype={'Individual':object,'colorIndex':int,'symbolIndex':int})
aPop=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/KNCnewPop1_IA.csv',names=['Individual','colorIndex','symbolIndex','CustomLabel'],dtype={'Individual':object,'colorIndex':int,'symbolIndex':int})
aPop_o=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/KNCnewPop1_IA_o.csv',names=['Individual','colorIndex','symbolIndex','CustomLabel'],dtype={'Individual':object,'colorIndex':int,'symbolIndex':int})
aPop_contam=pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/KNCnewPop1_IA_lc.csv',names=['Individual','colorIndex','symbolIndex','CustomLabel'],dtype={'Individual':object,'colorIndex':int,'symbolIndex':int})

In [ ]:
symbolVec = ['8','s','p','P','*','h','H','X','D','d',',','<','>','^','v','o','.','$a$','$b$','$c$','$d$','$e$',
'$f$','$g$','$h$','$i$','$j$','$k$','$l$','$m$','$n$','$o$','$p$','$q$','$r$','$s$','$t$','$u$','$v$','$w$','$x$','$y$','$z$','$A$','$B$','$C$','$D$','$E$',
'$F$','$G$','$H$','$I$','$J$','$K$','$L$','$M$','$N$','$O$','$P$','$Q$','$R$','$S$','$T$','$U$','$V$','$W$','$X$','$Y$','$Z$','+','x','1','2','3','4','|','_']
colorVec = [u'#1f77b4', u'#ff7f0e', u'#f72585', u'#d62728', u'#9467bd',u'#023e8a',
            u'#8c564b', u'#e377c2', u'#bcbd22', u'#17becf',u'#003049',u'#730220',u'#ffb703',u'#9e0059',u'#ffff3f',u'#2b9348',u'#219ebc',u'#ffd60a']

Counting samples

In [ ]:
aPop_counts = {row['Individual']: (pcaDat.Population == row['Individual']).sum() for _, row in aPop.iterrows()}
aPop_o_counts = {row['Individual']: (pcaDat.Population == row['Individual']).sum() for _, row in aPop_o.iterrows()}
refPop_counts = {row['Individual']: (pcaDat.Population == row['Individual']).sum() for _, row in refPop.iterrows()}

print("\nrefPop counts:")
for k, v in refPop_counts.items():
    print(f"  {k}: {v}")

print("\naPop counts:")
for k, v in aPop_counts.items():
    print(f"  {k}: {v}")

print("\naPop_o counts:")
for k, v in aPop_o_counts.items():
    print(f"  {k}: {v}")


In [ ]:
# existing counts
aPop_counts   = {row['Individual']: (pcaDat.Population == row['Individual']).sum() for _, row in aPop.iterrows()}
aPop_o_counts = {row['Individual']: (pcaDat.Population == row['Individual']).sum() for _, row in aPop_o.iterrows()}

# build prefix -> aPop full name (assumes prefixes are unique in aPop)
aPop_prefix_map = {ind[:5]: ind for ind in aPop_counts.keys()}

# add aPop_o counts into matching aPop by 5-char prefix
for ind_o, cnt_o in aPop_o_counts.items():
    prefix = ind_o[:5]
    if prefix in aPop_prefix_map:
        ind_main = aPop_prefix_map[prefix]
        aPop_counts[ind_main] += cnt_o

# (optional) print which ones were merged
print("\nMerged aPop_o into aPop (by first 5 chars):")
for ind_o, cnt_o in aPop_o_counts.items():
    prefix = ind_o[:5]
    if prefix in aPop_prefix_map:
        print(f"  {ind_o} (+{cnt_o}) -> {aPop_prefix_map[prefix]}")


Complete PCA

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D


fig = plt.figure(figsize=(16, 16), dpi=600)
ax = fig.add_subplot(111)

# Lists for custom handles and labels
custom_handles = []
custom_labels = []
apop_handles = []
apop_labels = []

# Set axis properties
ax.axis('equal')
ax.set_aspect('equal', 'box')

# Scatter plot for popListDat
for i, row in popListDat.iterrows():
    d = pcaDat[pcaDat.Population == row['Population']]
    ax.scatter(x=d['PC1'], y=-d['PC2'], c=[u'#7f7f7f'], label='', alpha=0.1)

# Scatter plot for refPop
for i, row in refPop.iterrows():
    d = pcaDat[pcaDat.Population == row['Individual']]
    scatter = plt.scatter(x=d['PC1'], y=-d['PC2'], c=colorVec[row['colorIndex']],
                           marker=symbolVec[row['symbolIndex']], label=row['Individual'], s=70)
    custom_handles.append(scatter)
    custom_labels.append(row['CustomLabel'])

# Scatter plot for aPop with number of samples in the legend
for i, row in aPop.iterrows():
    ind = row['Individual']
    d = pcaDat[pcaDat.Population == row['Individual']]
    scatter = plt.scatter(x=d['PC1'], y=-d['PC2'], c=colorVec[row['colorIndex']],
                           marker=symbolVec[row['symbolIndex']], label=row['Individual'], s=120, ec='k')
    apop_handles.append(scatter)
    # use merged counts (fallback to len(d) if key missing)
    n = aPop_counts.get(ind, len(d))
    apop_labels.append(f"{row['CustomLabel']} (n={n})")
    #apop_labels.append(f"{row['CustomLabel']} (n={len(d)})")


# Scatter plot for aPop_o with red edge color
for i, row in aPop_o.iterrows():
    d = pcaDat[pcaDat.Population == row['Individual']]
    scatter = plt.scatter(x=d['PC1'], y=-d['PC2'], c=colorVec[row['colorIndex']],
                           marker=symbolVec[row['symbolIndex']], label=row['Individual'], s=120, ec='red')

# Copy lists so you don't modify originals
handles = custom_handles.copy()
labels = custom_labels.copy()

# Pad to 22 entries (11 per column × 2 columns)
while len(handles) < 22:
    handles.append(Line2D([], [], linestyle="none"))  # invisible handle
    labels.append("")

legend1 = ax.legend(
    handles,
    labels,
    loc=(1.1, 0),
    fontsize=16,
    ncol=2,
    title="Published data"
)

# Add legend for refPop
#legend1 = ax.legend(custom_handles, custom_labels, loc=(1.1, 0), fontsize=16, ncol=2, title="Published data")
plt.setp(legend1.get_title(), fontsize=18)  # Set title fontsize for the first legend
ax.add_artist(legend1)  # Add this legend first

# Add legend for aPop
legend2 = ax.legend(apop_handles, apop_labels, loc=(1.1, 0.5), fontsize=16, ncol=1, title="New data")
plt.setp(legend2.get_title(), fontsize=18)  # Set title fontsize for the second legend

# Add a rectangle frame with the specified limits
frame_xlim = (-0.0014, 0.0013)
frame_ylim = (-0.0019, 0.0003)
frame = patches.Rectangle(
    (frame_xlim[0], frame_ylim[0]),  # Bottom-left corner
    frame_xlim[1] - frame_xlim[0],  # Width
    frame_ylim[1] - frame_ylim[0],  # Height
    linewidth=2, edgecolor='black', facecolor='none', linestyle='-'
)
ax.add_patch(frame)

# Set labels and show plot
plt.xlabel('PC1 (0.71%)', fontsize=20)
plt.ylabel('PC2 (0.40%)', fontsize=20)
ax.tick_params(axis='both', labelsize=16)
fig.savefig("PCA.svg", format='svg', bbox_inches='tight')
plt.show()


Zoom-in PCA

In [ ]:
fig=plt.figure(figsize=(12,12),dpi=600, facecolor="white")
ax=fig.add_subplot(111, facecolor="white")
# Lists for custom handles and labels
custom_handles = []
custom_labels = []
ax.axis('equal')
ax.set_aspect('equal', 'box')
ax.set(xlim=(-0.0014,0.0013),ylim=(-0.0019,0.0003))
# Remove ticks but keep the frame
ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
# Scatter plot for popListDat
for i, row in popListDat.iterrows():
    d = pcaDat[pcaDat.Population == row['Population']]
    ax.scatter(x=d['PC1'], y=-d['PC2'], c=[u'#7f7f7f'], label='', alpha=0.1)

# Scatter plot for refPop
for i, row in refPop.iterrows():
    d = pcaDat[pcaDat.Population == row['Individual']]
    scatter = plt.scatter(x=d['PC1'], y=-d['PC2'], c=colorVec[row['colorIndex']],
                           marker=symbolVec[row['symbolIndex']], label=row['Individual'], s=70)
    #custom_handles.append(scatter)
    #custom_labels.append(row['CustomLabel'])

# Scatter plot for aPop
for i, row in aPop.iterrows():
    d = pcaDat[pcaDat.Population == row['Individual']]
    scatter = plt.scatter(x=d['PC1'], y=-d['PC2'], c=colorVec[row['colorIndex']],
                           marker=symbolVec[row['symbolIndex']], label=row['Individual'], s=120, ec='k')
    custom_handles.append(scatter)
    custom_labels.append(row['CustomLabel'])

# Scatter plot for aPop_o with red edge color
for i, row in aPop_o.iterrows():
    d = pcaDat[pcaDat.Population == row['Individual']]
    scatter = plt.scatter(x=d['PC1'], y=-d['PC2'], c=colorVec[row['colorIndex']],
                           marker=symbolVec[row['symbolIndex']], label=row['Individual'], s=120, ec='#eb5e28')
    #custom_handles.append(scatter)
    #custom_labels.append(row['CustomLabel'])

# Add legend for all combined handles and labels
#ax.legend(custom_handles, custom_labels, loc=(1.1, 0), fontsize=16, ncol=2)
#plt.xlabel('PC1')
#plt.ylabel('PC2')
fig.savefig("PCA_zoomin.svg", format='svg', bbox_inches='tight')
plt.show()